# Passo 2 (controllo del 26B) e Passo 3b — esecuzione e stato della generazione

Notebook di esecuzione delle generazioni con **tutte le scelte già fatte** (docs/decisions.md, voci 118-132). Le
generazioni girano **in background** (`experiments/run_batch.py`): il notebook si può chiudere e riaprire, la cella
*Stato* mostra a che punto è la generazione; una run interrotta si riprende rilanciando la stessa cella (`--resume`).

| | Passo 2 — controllo del 26B (voci 131-132) | Passo 3b — esperimento principale (voce 132) |
|---|---|---|
| esercizi | 20 di sviluppo | **54 di valutazione** (34 corpus + 20 De Bari) |
| condizioni | bm25 k = 3 | zero_shot, static, random, bm25 k = 2 e 3, hybrid (dense tolto, voce 144) |
| ripetizioni | 2 | 3 |
| generazioni | 80 | 1.938 |
| config | `experiments/configs/dev_g26.yaml` | `experiments/configs/passo3b.yaml` |
| analisi | `experiments/analyze_g26.py` (regola registrata) | da scrivere e testare PRIMA delle run |

Scelte fisse: istruzioni v5 (PlantUML) e v5 + esempi indentati (compatto); esempi dal file congelato
`data/retrieval_frozen/retrievals_3b.jsonl`; random con seme per esercizio e ripetizione; temperatura 0,3, top_p
0,95, top_k 64; contesto 32768, max_tokens 4096; ragionamento spento (`thinking: none`).

**Sicurezze**: tutti gli interruttori sono a `False`. Il 3b parte con il via dell'utente (voce 149, prima delle
risposte dei relatori): i risultati non si guardano finché lo script di analisi del 3b non è scritto e testato.

In [ ]:
# ---- interruttori e parametri (lasciare False salvo decisione esplicita) ----
MODELLO = "G26"          # "G26" = Gemma 4 26B A4B QAT (se il passo 2 lo adotta) | "G12" = Gemma 4 12B QAT
CARICA_MODELLO = True   # carica il modello in LM Studio con il comando registrato (scarica quello caricato)
ESEGUI_PASSO2 = False     # avvia le 80 generazioni del controllo del 26B (voce 132)
SBLOCCA_3B = False       # avvia il 3b (voce 149: via dell'utente; non guardare i risultati prima dello script di analisi)

import json, os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # percorsi dei config e degli script relativi alla radice del repo
sys.path[:0] = [str(ROOT / "experiments"), str(ROOT / "generation"), str(ROOT / "retrieval"), str(ROOT / "corpus")]
import progress, run_batch

MODELS = {"G26": ("google/gemma-4-26b-a4b-qat", "--context-length 32768 --gpu 0.5 -y"),
          "G12": ("google/gemma-4-12b-qat", "--context-length 32768 --gpu max -y")}
PASSO2 = ("passo2_g26", [("experiments/configs/dev_g26.yaml", "P-G"), ("experiments/configs/dev_g26.yaml", "C-G")])
PASSO3B = (f"passo3b_{MODELLO.lower()}", [("experiments/configs/passo3b.yaml", f"{f}-{s}-{MODELLO}")
                                          for f in "PC" for s in ("corpus", "testset")])

def sh(cmd):
    """Esegue uno script del repo con lo stesso Python del notebook e ne stampa l'uscita."""
    r = subprocess.run([sys.executable, *cmd.split()], cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
    print(r.stdout[-6000:], r.stderr[-3000:] if r.returncode else "", f"[codice {r.returncode}]")
    return r.returncode

print("Python:", sys.executable)
print("Modello scelto:", MODELLO, MODELS[MODELLO][0])

## 1. Controlli preliminari
Ambiente, file congelato (sha256), ruoli congelati (tag `evalset-v1`), GPU, modello caricato in LM Studio con il
contesto giusto. Nessuna generazione.

In [ ]:
import urllib.request
ok = True
if ".venv" not in sys.executable:
    print("ATTENZIONE: il kernel non usa .venv (docs/STATUS.md, 'Ambiente della tesi'): le run del 3b vanno fatte lì")
from prompt_builder import PromptBuilder
b = PromptBuilder(retrieval_source="frozen")
print("file congelato: sha256", b.frozen_sha256[:16], "verificato,", len(b.frozen), "righe")
same = subprocess.run(["git", "diff", "--quiet", "evalset-v1", "--", "corpus/roles.yaml"], cwd=ROOT).returncode == 0
print("roles.yaml invariato da evalset-v1:", same); ok &= same
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
try:
    models = json.load(urllib.request.urlopen("http://localhost:1234/api/v1/models", timeout=10))
    loaded = {i["id"]: i["config"]["context_length"] for m in models.get("models", models.get("data", []))
              for i in m.get("loaded_instances", [])}
    want = MODELS[MODELLO][0]
    print("LM Studio, modelli caricati:", loaded)
    if loaded.get(want) != 32768:
        print(f"ATTENZIONE: serve {want} con contesto 32768 (cella 2)"); ok = False
except OSError as e:
    print("LM Studio non raggiungibile:", e); ok = False
print("\nCONTROLLI", "SUPERATI" if ok else "NON superati: non avviare le generazioni")

## 2. Caricare il modello in LM Studio
Comandi registrati: 26B con metà degli strati sulla GPU (voce 131), 12B interamente sulla GPU. Scarica i modelli
caricati. Con `CARICA_MODELLO = False` stampa soltanto il comando.

In [ ]:
key, opts = MODELS[MODELLO]
cmd = f"lms load {key} {opts}"
print("comando:", cmd)
if CARICA_MODELLO:
    print(subprocess.run("lms unload --all", shell=True, capture_output=True, text=True, encoding="utf-8", errors="replace").stdout)
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True, encoding="utf-8", errors="replace").stdout[-1500:])


## 3. Passo 2 — controllo del 26B sul dev set (80 generazioni, circa 40 minuti)
Regola registrata PRIMA della run (voce 132): il 26B diventa il modello del 3b se, in entrambi i formati, dR ≥ −0,03,
dVc ≥ −2, dM_primaria ≥ −0,03, meno di 2 troncamenti, e la stima del 3b è ≤ 30 ore. Richiede il 26B caricato.

In [ ]:
nome, runs = PASSO2
if ESEGUI_PASSO2:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid}); log: data/results/batch/{nome}/batch.log")
else:
    print("ESEGUI_PASSO2 = False: nessuna generazione avviata")

### Stato del passo 2 (rieseguire la cella quando si vuole; `watch` aggiorna da solo ogni minuto)

In [ ]:
try:
    print(progress.report(PASSO2[0]))
except FileNotFoundError:
    print("batch non ancora avviato")
# progress.watch(PASSO2[0], every_s=60, max_minutes=60)   # aggiornamento automatico

### Analisi del passo 2 (a run complete): applica la regola e scrive `data/results/generation/dev_g26_analysis/summary.md`

In [ ]:
try:
    st = progress.status(PASSO2[0])
except FileNotFoundError:
    st = None
if st and st["fatte"] == st["attese"] and not st["vivo"]:
    sh("experiments/analyze_g26.py")
else:
    print("passo 2 non completo: l'analisi si esegue a run finite", f"({st['fatte']}/{st['attese']})" if st else "")

## 3 bis. Leva "esempi senza istruzioni per lo studente" (voce 138, 32 generazioni, circa 15 minuti)
Come il passo 2: le 8 query di sviluppo toccate dalla vista `clean_v1` (11 frasi tolte da 6 esercizi QUANDO compaiono
come esempio), 26B, configurazione di lavoro, 2 ripetizioni × 2 formati. Regola registrata PRIMA della run (non
inferiorità): si adotta per tutto il 3b se, in entrambi i formati, dF1_relazioni ≥ −0,03, dR ≥ −0,03, dVc ≥ −1 e meno di
2 troncamenti. Richiede il 26B caricato (cella 2).

In [ ]:
ESEGUI_LEVA_EXVIEW = False   # mettere True per avviare le 32 generazioni
LEVA_EXVIEW = ("leva_exview", [("experiments/configs/dev_g26ev.yaml", "P-G"), ("experiments/configs/dev_g26ev.yaml", "C-G")])

nome, runs = LEVA_EXVIEW
if ESEGUI_LEVA_EXVIEW:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid}); log: data/results/batch/{nome}/batch.log")
else:
    print("ESEGUI_LEVA_EXVIEW = False: nessuna generazione avviata")

### Stato della leva (rieseguire la cella quando si vuole)

In [ ]:
try:
    print(progress.report(LEVA_EXVIEW[0]))
except FileNotFoundError:
    print("batch non ancora avviato")
# progress.watch(LEVA_EXVIEW[0], every_s=60, max_minutes=30)   # aggiornamento automatico

### Analisi della leva (a run complete): applica la regola e scrive `data/results/generation/dev_g26ev_analysis/summary.md`

In [ ]:
try:
    st = progress.status(LEVA_EXVIEW[0])
except FileNotFoundError:
    st = None
if st and st["fatte"] == st["attese"] and not st["vivo"]:
    sh("experiments/analyze_exview.py")
else:
    print("leva non completa: l'analisi si esegue a run finite", f"({st['fatte']}/{st['attese']})" if st else "")

## 3 ter. Passo 2 rifatto con il retriever hybrid (voce 140, 80 generazioni, circa 45 minuti)
Configurazione di lavoro del 3b (26B, PlantUML v5, compatto v5 + esempi indentati, esempi puliti `clean_v1`, file
congelato) sui 20 esercizi di sviluppo, con la condizione **hybrid** k = 3. La baseline bm25 è già disponibile (dev_g26
e dev_g26ev, prompt identici): si generano solo le risposte hybrid. Regola registrata PRIMA della run (non inferiorità):
hybrid resta nel 3b se, in entrambi i formati, dF1_relazioni ≥ −0,03, dR ≥ −0,03, dVc ≥ −2 e meno di 2 troncamenti;
altrimenti decide l'utente. Richiede il 26B caricato (cella 2).

In [ ]:
ESEGUI_PASSO2_HYBRID = False   # mettere True per avviare le 80 generazioni
PASSO2_HYBRID = ("passo2_hybrid", [("experiments/configs/dev_g26hy.yaml", "P-G"), ("experiments/configs/dev_g26hy.yaml", "C-G")])

nome, runs = PASSO2_HYBRID
if ESEGUI_PASSO2_HYBRID:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid}); log: data/results/batch/{nome}/batch.log")
else:
    print("ESEGUI_PASSO2_HYBRID = False: nessuna generazione avviata")

### Stato del passo 2 con hybrid (rieseguire la cella quando si vuole)

In [ ]:
try:
    print(progress.report(PASSO2_HYBRID[0]))
except FileNotFoundError:
    print("batch non ancora avviato")
# progress.watch(PASSO2_HYBRID[0], every_s=60, max_minutes=60)   # aggiornamento automatico

### Analisi (a run complete): applica la regola e scrive `data/results/generation/dev_g26hy_analysis/summary.md`

In [ ]:
try:
    st = progress.status(PASSO2_HYBRID[0])
except FileNotFoundError:
    st = None
if st and st["fatte"] == st["attese"] and not st["vivo"]:
    sh("experiments/analyze_hybrid26.py")
else:
    print("run non completa: l'analisi si esegue a run finite", f"({st['fatte']}/{st['attese']})" if st else "")

## 3 quater. Passo 2 rifatto con il retriever dense (voce 142, 80 generazioni, circa 45 minuti)
Come la 3 ter, con la condizione **dense** k = 3 (MiniLM). Baseline bm25 già disponibile. Regola registrata PRIMA della
run, identica a quella di hybrid: dense resta nel 3b se, in entrambi i formati, dF1_relazioni ≥ −0,03, dR ≥ −0,03,
dVc ≥ −2 e meno di 2 troncamenti; altrimenti decide l'utente. Nel report anche hybrid, solo descrittivo.

In [ ]:
ESEGUI_PASSO2_DENSE = False   # mettere True per avviare le 80 generazioni
PASSO2_DENSE = ("passo2_dense", [("experiments/configs/dev_g26de.yaml", "P-G"), ("experiments/configs/dev_g26de.yaml", "C-G")])

nome, runs = PASSO2_DENSE
if ESEGUI_PASSO2_DENSE:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid}); log: data/results/batch/{nome}/batch.log")
else:
    print("ESEGUI_PASSO2_DENSE = False: nessuna generazione avviata")

### Stato del passo 2 con dense (rieseguire la cella quando si vuole)

In [ ]:
try:
    print(progress.report(PASSO2_DENSE[0]))
except FileNotFoundError:
    print("batch non ancora avviato")
# progress.watch(PASSO2_DENSE[0], every_s=60, max_minutes=60)   # aggiornamento automatico

### Analisi (a run complete): applica la regola e scrive `data/results/generation/dev_g26de_analysis/summary.md`

In [ ]:
try:
    st = progress.status(PASSO2_DENSE[0])
except FileNotFoundError:
    st = None
if st and st["fatte"] == st["attese"] and not st["vivo"]:
    sh("experiments/analyze_dense26.py")
else:
    print("run non completa: l'analisi si esegue a run finite", f"({st['fatte']}/{st['attese']})" if st else "")

## 3 quinquies. Leva "attributi del compatto" (istruzioni v5b, voce 146, 16 generazioni, circa 10 minuti)
Solo JSON compatto, 8 esercizi di sviluppo × 2 ripetizioni (i 6 con risposte senza attributi + EUScienceConnect e
Facepage di controllo), configurazione di lavoro (vista clean_v1, bm25 k = 3, file congelato). Baseline v5: 7
scheletri su 16. Regola registrata PRIMA della run: v5b adottata per il compatto del 3b se scheletri ≤ 3, sui 2 di
controllo recall e precision degli attributi non scendono più di 0,05, e sulle 16 risposte dF1_relazioni ≥ −0,03,
dR ≥ −0,03, dM_primaria ≥ −0,03, dVc ≥ −1 e meno di 2 troncamenti.

In [ ]:
ESEGUI_LEVA_V5B = False   # mettere True per avviare le 16 generazioni
LEVA_V5B = ("leva_v5b", [("experiments/configs/dev_g26b.yaml", "C-G")])

nome, runs = LEVA_V5B
if ESEGUI_LEVA_V5B:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid}); log: data/results/batch/{nome}/batch.log")
else:
    print("ESEGUI_LEVA_V5B = False: nessuna generazione avviata")

### Stato della leva v5b (rieseguire la cella quando si vuole)

In [ ]:
try:
    print(progress.report(LEVA_V5B[0]))
except FileNotFoundError:
    print("batch non ancora avviato")
# progress.watch(LEVA_V5B[0], every_s=60, max_minutes=30)   # aggiornamento automatico

### Analisi (a run complete): applica la regola e scrive `data/results/generation/dev_g26b_analysis/summary.md`

In [ ]:
try:
    st = progress.status(LEVA_V5B[0])
except FileNotFoundError:
    st = None
if st and st["fatte"] == st["attese"] and not st["vivo"]:
    sh("experiments/analyze_v5b.py")
else:
    print("run non completa: l'analisi si esegue a run finite", f"({st['fatte']}/{st['attese']})" if st else "")

## 3 sexies. Leva "attributes sempre presente" (istruzioni v5c, voce 150, 40 generazioni, circa 25 minuti)
Solo JSON compatto, 20 esercizi di sviluppo × 2 ripetizioni, configurazione di lavoro (vista clean_v1, bm25 k = 3,
file congelato). v5c: istruzioni con "attributes" obbligatoria in ogni classe non enum ([] se il testo non ne dà) ed
esempi con `"attributes": []` esplicita. Baseline v5: 7 scheletri su 40. Regola registrata PRIMA della run: v5c
adottata per il compatto del 3b se scheletri ≤ 3, recall degli attributi non scende, precision degli attributi
≥ −0,05, dF1_relazioni, dR e dM_primaria ≥ −0,03, dVc ≥ −2 e meno di 2 troncamenti. Se adottata, si rifà la parte
compatta del 3b (969 generazioni, circa 10 ore). Richiede il 26B caricato (sezione 2).

In [ ]:
ESEGUI_LEVA_V5C = False   # mettere True per avviare le 40 generazioni
LEVA_V5C = ("leva_v5c", [("experiments/configs/dev_g26c.yaml", "C-G")])

nome, runs = LEVA_V5C
if ESEGUI_LEVA_V5C:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid}); log: data/results/batch/{nome}/batch.log")
else:
    print("ESEGUI_LEVA_V5C = False: nessuna generazione avviata")

### Stato della leva v5c (rieseguire la cella quando si vuole)

In [ ]:
try:
    print(progress.report(LEVA_V5C[0]))
except FileNotFoundError:
    print("batch non ancora avviato")
# progress.watch(LEVA_V5C[0], every_s=60, max_minutes=40)   # aggiornamento automatico

### Analisi (a run complete): applica la regola e scrive `data/results/generation/dev_g26c_analysis/summary.md`

In [ ]:
try:
    st = progress.status(LEVA_V5C[0])
except FileNotFoundError:
    st = None
if st and st["fatte"] == st["attese"] and not st["vivo"]:
    sh("experiments/analyze_v5c.py")
else:
    print("run non completa: l'analisi si esegue a run finite", f"({st['fatte']}/{st['attese']})" if st else "")

## 4. Passo 3b — esperimento principale (1.938 generazioni)
Quattro run per il modello scelto: `passo3b__{P,C}-{corpus,testset}-<modello>`. Prima un **dry run** (nessuna chiamata
al modello: lunghezze dei prompt e finestre di contesto), poi l'avvio, che richiede `SBLOCCA_3B = True`.

In [ ]:
nome, runs = PASSO3B
print("run del 3b:", [k for _, k in runs])
print("generazioni attese:", sum(progress.expected_calls(c, k) for c, k in runs))
# dry run (nessuna chiamata al modello), scritto in data/results/generation/dry_run/ (non versionato)
for c, k in runs:
    sh(f"experiments/run_experiment.py {c} --configuration {k} --dry-run")

In [ ]:
if SBLOCCA_3B:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid}); log: data/results/batch/{nome}/batch.log")
else:
    print("SBLOCCA_3B = False: il 3b non parte (relatori, domande 8, 11, 14, 16; script di analisi del 3b)")

### Stato del 3b

In [ ]:
try:
    print(progress.report(PASSO3B[0]))
except FileNotFoundError:
    print("batch non ancora avviato")
# progress.watch(PASSO3B[0], every_s=120, max_minutes=120)   # aggiornamento automatico

## 4 bis. Passo 3b sui 26 esercizi di sviluppo — analisi SECONDARIA (voce 151, 936 generazioni, circa 7 ore)
Stessa configurazione del 3b (`experiments/configs/passo3b_sviluppo.yaml`), sui 26 esercizi di sviluppo (20 del dev
set + 6 dei piloti). Le scelte della configurazione sono state fatte su questi esercizi (con bm25): i risultati si
riportano A PARTE, mai insieme ai 54 di valutazione. Run: `passo3b_sviluppo__{P,C}-sviluppo-G26`. Richiede il 26B
caricato (sezione 2).

In [ ]:
ESEGUI_3B_SVILUPPO = True   # mettere True per avviare le 936 generazioni
PASSO3B_SVILUPPO = ("passo3b_sviluppo", [("experiments/configs/passo3b_sviluppo.yaml", f"{f}-sviluppo-G26") for f in "PC"])

nome, runs = PASSO3B_SVILUPPO
print("generazioni attese:", sum(progress.expected_calls(c, k) for c, k in runs))
if ESEGUI_3B_SVILUPPO:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid}); log: data/results/batch/{nome}/batch.log")
else:
    print("ESEGUI_3B_SVILUPPO = False: nessuna generazione avviata")

### Stato del 3b sui 26 di sviluppo

In [ ]:
try:
    print(progress.report(PASSO3B_SVILUPPO[0]))
except FileNotFoundError:
    print("batch non ancora avviato")
# progress.watch(PASSO3B_SVILUPPO[0], every_s=120, max_minutes=480)   # aggiornamento automatico

## 4 ter. Controllo "esercizio stesso come esempio" (oracle_self, voce 162, 216 generazioni, circa 1,5 ore)
SOLO ANALISI: ogni esercizio di validazione ha come unico esempio se stesso con la soluzione. Varianti `standard`
(con "do not copy their classes") e `libero` (senza). Regola: completezza (variante libero) >= 0,80 = il modello sa
usare un esempio perfetto (limite: il corpus); < 0,60 = non lo usa (limite: modello o istruzioni). Richiede il 26B.

In [ ]:
ESEGUI_ORACLE_SELF = False   # mettere True per avviare le 216 generazioni
ORACLE_SELF = ("oracle_self", [("experiments/configs/oracle_self.yaml", f"{f}-{s}-{v}")
                                for f in "PC" for s in ("corpus", "testset") for v in ("standard", "libero")])

nome, runs = ORACLE_SELF
if ESEGUI_ORACLE_SELF:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid})")
else:
    print("ESEGUI_ORACLE_SELF = False: nessuna generazione avviata")
try:
    print(progress.report(nome))
except FileNotFoundError:
    print("batch non ancora avviato")

In [ ]:
# analisi (a run complete): scrive data/results/generation/oracle_self_analysis/summary.md
sh("experiments/analyze_oracle_self.py")

## 4 quater. Controllo "esempio parzialmente simile" (oracle_partial, voce 163, 162 generazioni, circa 50 minuti)
SOLO ANALISI, PlantUML: l'esempio e' l'esercizio stesso con la soluzione ridotta al 25 / 50 / 75 % delle classi.
Soglia di utilita' = il livello piu' basso con guadagno su zero_shot >= 0,05. Lanciare dopo la fine di oracle_self.

In [ ]:
ESEGUI_ORACLE_PARTIAL = False   # mettere True per avviare le 162 generazioni
ORACLE_PARTIAL = ("oracle_partial", [("experiments/configs/oracle_partial.yaml", f"P-{s}-{l}")
                                      for s in ("corpus", "testset") for l in (25, 50, 75)])

nome, runs = ORACLE_PARTIAL
if ESEGUI_ORACLE_PARTIAL:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid})")
try:
    print(progress.report(nome))
except FileNotFoundError:
    print("batch non ancora avviato")
# a run complete: sh("experiments/analyze_oracle_partial.py")

## 4 quinquies. Controllo "piu' esempi insieme" (oracle_mix, voce 164, 108 generazioni, circa 35 minuti)
SOLO ANALISI, PlantUML: soluzione al 50 % + 2 esempi bm25 (diluizione) oppure + la meta' complementare (somma).

In [ ]:
ESEGUI_ORACLE_MIX = False   # mettere True per avviare le 108 generazioni
ORACLE_MIX = ("oracle_mix", [("experiments/configs/oracle_mix.yaml", f"P-{s}-{v}")
                              for s in ("corpus", "testset") for v in ("diluito", "complemento")])
nome, runs = ORACLE_MIX
if ESEGUI_ORACLE_MIX:
    print("pid", run_batch.launch(nome, runs))
try:
    print(progress.report(nome))
except FileNotFoundError:
    print("batch non ancora avviato")
# a run complete: sh("experiments/analyze_oracle_mix.py")

## 4 sexies. Controllo "esempi senza attributi" (voce 167, 216 generazioni, circa 1 ora e 45)
SOLO ANALISI. A: esercizio stesso senza attributi; B: esempi bm25 del 3b senza attributi (causa degli scheletri).

In [ ]:
ESEGUI_ORACLE_ATTR = False   # mettere True per avviare le 216 generazioni
ORACLE_ATTR = ("oracle_attr", [(f"experiments/configs/{n}.yaml", f"{f}-{s}")
                                for n in ("oracle_attr_bm25", "oracle_attr_self") for f in "CP" for s in ("corpus", "testset")])
nome, runs = ORACLE_ATTR
if ESEGUI_ORACLE_ATTR:
    print("pid", run_batch.launch(nome, runs))
try:
    print(progress.report(nome))
except FileNotFoundError:
    print("batch non ancora avviato")
# a run complete: sh("experiments/analyze_oracle_attr.py")

## 5. Fermare un batch
`run_batch.stop(nome)` ferma **alla fine della run in corso**; `run_batch.stop(nome, now_=True)` termina subito il
processo. In entrambi i casi le chiamate già fatte restano (cache e manifest): rilanciando la cella di avvio il batch
riprende dalle mancanti.

In [ ]:
run_batch.stop("passo3b_sviluppo", now_=True)
# run_batch.stop(PASSO2[0])
#run_batch.stop(PASSO3B[0], now_=True)

## 6. Confronto visivo di un esercizio: ground truth, PlantUML e JSON compatto
Per l'esercizio e la ripetizione scelti: il **diagramma di riferimento** e le **risposte delle run indicate**, una accanto
all'altra, disegnate tutte allo stesso modo (PlantUML canonico → SVG con `plantuml.jar` locale; classi **verdi** = presenti
nel GT, **arancioni** = in più). Per ogni risposta: livello di validità, V / Vc / J / R / M e l'elenco delle relazioni
(corretta, tipo sbagliato, direzione invertita, molteplicità diverse, mancante, in più); le risposte non valide mostrano
il testo grezzo. La pagina si salva anche in `data/results/generation/confronti/` per aprirla nel browser.

- Passo 2: `RUNS = ["dev_g26__P-G", "dev_g26__C-G"]`; per confrontare anche il 12B aggiungere `"dev_v5__P-G"` e
  `"dev_c5indent__C-G"`.
- 3b: per esempio `RUNS = ["passo3b__P-corpus-G26", "passo3b__C-corpus-G26"]` con `CONDIZIONE = "bm25"` e `K = 3`
  (senza condizione compaiono tutte le condizioni dell'esercizio).

In [ ]:
import compare_view as cv
from IPython.display import display

RUNS = ["dev_g26__P-G", "dev_g26__C-G"]   # + "dev_v5__P-G", "dev_c5indent__C-G" per il confronto con il 12B
ESERCIZIO = "Boeing"
RIPETIZIONE = 0
CONDIZIONE, K = None, None                 # nel 3b: es. "bm25", 3

print("esercizi disponibili:", ", ".join(cv.exercises(RUNS)))
display(cv.show(ESERCIZIO, RUNS, RIPETIZIONE, CONDIZIONE, K))

## 6 bis. Confronto bm25 / hybrid con il ground truth (strumento interattivo)
Pagina nel browser con, per ogni esercizio di sviluppo, formato e ripetizione, **GT | bm25 | hybrid** affiancati (26B,
configurazione di lavoro con esempi puliti). Menu per esercizio, formato e ripetizione; frecce ◀ ▶ (o tasti ← →);
scheda **Indice** con l'F1 delle relazioni per esercizio e retriever (clic su una riga per aprire il confronto). Per
ogni risposta: livello, F1 di relazioni / classi / generalizzazioni, R, J, M (in verde la migliore delle due),
relazioni classificate ed esempi usati nel prompt. Clic su un diagramma per ingrandirlo.

In [ ]:
import os
if sh("experiments/compare_retrievers.py") == 0:
    page = ROOT / "data" / "results" / "generation" / "confronto_bm25_hybrid" / "index.html"
    print("aperta nel browser:", page)
    os.startfile(page)

## 6 ter. Confronto di alcuni esercizi del 3b in tutte le condizioni
Pagina HTML: per ogni esercizio scelto, il ground truth accanto alle risposte delle sei condizioni (zero_shot,
static k2, random k3, bm25 k2 e k3, hybrid k3), per formato e ripetizione, con metriche, relazioni classificate ed
esempi del prompt con la loro somiglianza (Jt) all'esercizio. Scheda *Indice*: F1 delle relazioni medio per
condizione. Salvata in `data/results/generation/confronto_3b/index.html` (non versionata).

In [ ]:
import os
ESERCIZI_3B = ["tutti"]   # ["tutti"] = i 54 di valutazione (circa 1 minuto); oppure una lista di id, es. ["Gym", "RealEstateAgency"]

if sh("experiments/compare_3b.py --esercizi " + " ".join(ESERCIZI_3B)) == 0:
    page = ROOT / "data" / "results" / "generation" / "confronto_3b" / "index.html"
    print("aperta nel browser:", page)
    os.startfile(page)   # Windows: browser predefinito

## 7. Tutti i risultati di una run: report di revisione completo
Pagina HTML con **tutti** gli esercizi della run: un indice con V / Vc / J / R / M per esercizio e condizione, poi per
ogni esercizio il ground truth e tutte le risposte (tutte le ripetizioni e i formati), con le relazioni classificate.
Si apre nel browser predefinito. Salvata in `data/results/generation/<nome>_review/index.html` (non versionata).
Le metriche complessive e l'esito della regola restano nel `summary.md` dell'analisi (cella del passo 2).

In [ ]:
import os
# 3b: quattro run (formato x split); con tutte le condizioni sono 1.938 risposte (pagina di circa 50-60 MB): filtrare.
REPORT_RUNS = ["passo3b__P-corpus-G26", "passo3b__P-testset-G26", "passo3b__C-corpus-G26", "passo3b__C-testset-G26"]
REPORT_FILTRI = ["cond=bm25", "k=3"]             # una condizione alla volta: cond=zero_shot | static | random | bm25 | hybrid (+ k=2/3)
                                                 # oppure ["q=Gym"] per un esercizio, ["rep=0"] per una ripetizione
REPORT_NOME = "passo3b_" + "_".join(f.replace("=", "") for f in REPORT_FILTRI)
# passo 2: REPORT_RUNS = ["dev_g26__P-G", "dev_g26__C-G"]; REPORT_FILTRI = []; REPORT_NOME = "dev_g26"

args = " ".join(["experiments/review_report.py --runs", *REPORT_RUNS, "--name", REPORT_NOME,
                 *[f"--filter {f}" for f in REPORT_FILTRI]])
if sh(args) == 0:
    page = ROOT / "data" / "results" / "generation" / f"{REPORT_NOME}_review" / "index.html"
    print("aperta nel browser:", page)
    os.startfile(page)   # Windows: browser predefinito

## 8. Dopo le run
- Passo 2: cella di analisi qui sopra; esito da registrare in docs/decisions.md (e `MODELLO` di conseguenza).
- Passo 3b: lo script di analisi (confronti appaiati sui 54 e sui 52, static contro retrieval su 53, pertinenza e
  qualità tra esercizi ed entro esercizio; voci 129-130) va scritto e testato PRIMA di avviare le run.
- Revisione visiva di un campione: `python experiments/review_report.py --runs <run> --name <nome>`.